# Setup & dataloading

In [1]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from glob import glob
import re
import warnings
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')

# Paper-quality plot settings
plt.rcParams.update({
    'font.size': 16,
    'axes.labelsize': 16,
    'axes.titlesize': 16,
    'legend.fontsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'figure.figsize': (8, 8),
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'lines.linewidth': 2,
    'axes.grid': False,
    'grid.alpha': 0.3,
    'font.family': 'arial',
    'legend.frameon': False
})

# Output directory for plots
PLOT_DIR = Path('plots/mnist_rmsProp/')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Plots will be saved to: {PLOT_DIR.absolute()}")

Plots will be saved to: /Users/sambt/iaifi/sv3/claude_experiments/plots/mnist_rmsProp


In [2]:
# Load all experiment results and combine into single DataFrame
RESULTS_DIR = Path('../experiment_results/mnist_scan_varyRtol/')

all_dfs = []
file_pattern = '*.pkl'

for filepath in sorted(RESULTS_DIR.glob(file_pattern)):
    # Extract batch_size and width from filename
    match = re.search(r'bs(\d+)_width(\d+)', filepath.name)
    if match:
        bs_from_name = int(match.group(1))
        width_from_name = int(match.group(2))
        
        with open(filepath, 'rb') as f:
            df_temp = pickle.load(f)
        
        # Ensure batch_size and mlp_width columns exist
        if 'batch_size' not in df_temp.columns:
            df_temp['batch_size'] = bs_from_name
        if 'mlp_width' not in df_temp.columns:
            df_temp['mlp_width'] = width_from_name
            
        all_dfs.append(df_temp)
        #print(f"Loaded {filepath.name}: {len(df_temp)} rows")

df = pd.concat(all_dfs, ignore_index=True)
print(f"\nTotal: {len(df)} experiment runs")
print(f"\nColumns: {df.columns.tolist()}")


Total: 421 experiment runs

Columns: ['batch_size', 'k_fraction', 'k', 'lr', 'rtol', 'mlp_width', 'losses', 'svd_info', 'svd_mode', 'rmsProp', 'alpha_rmsProp', 'optimizer']


In [3]:
# Helper functions
def get_final_loss(row, loss_type='val'):
    """Get final epoch loss."""
    return row['losses'][loss_type][-1]

def get_final_acc(row, acc_type='val_acc'):
    """Get final epoch accuracy."""
    if acc_type in row['losses']:
        return row['losses'][acc_type][-1]
    return np.nan

def get_best_acc(row, acc_type='val_acc'):
    """Get best epoch accuracy."""
    if acc_type in row['losses']:
        return max(row['losses'][acc_type])
    return np.nan

def get_best_loss(row, loss_type='val'):
    """Get best epoch loss."""
    return min(row['losses'][loss_type])

def get_loss_curve(row, loss_type='val'):
    """Get per-epoch loss curve."""
    return np.array(row['losses'][loss_type])

def get_acc_curve(row, acc_type='val_acc'):
    """Get per-epoch accuracy curve."""
    if acc_type in row['losses']:
        return np.array(row['losses'][acc_type])
    return None

def get_svd_info(row):
    """Get SVD singular values and counts."""
    if row['svd_info'] is None or (isinstance(row['svd_info'], float) and pd.isna(row['svd_info'])):
        return None, None
    return row['svd_info'].get('svs'), row['svd_info'].get('num_nonzero_svs')

def sliding_average(data, window=10):
    """Compute sliding average for smoothing."""
    return np.convolve(data, np.ones(window)/window, mode='valid')

# Add derived columns
df['final_val_loss'] = df.apply(lambda r: get_final_loss(r, 'val'), axis=1)
df['final_train_loss'] = df.apply(lambda r: get_final_loss(r, 'train'), axis=1)
df['final_val_acc'] = df.apply(lambda r: get_final_acc(r, 'val_acc'), axis=1)
df['final_train_acc'] = df.apply(lambda r: get_final_acc(r, 'train_acc'), axis=1)
df['best_val_acc'] = df.apply(lambda r: get_best_acc(r, 'val_acc'), axis=1)
df['best_train_acc'] = df.apply(lambda r: get_best_acc(r, 'train_acc'), axis=1)
df['best_val_loss'] = df.apply(lambda r: get_best_loss(r, 'val'), axis=1)

In [6]:
# Separate SVD and baseline results
df_svd = df[df['optimizer'] == 'SVD'].copy()
df_baseline = df[df['optimizer'] != 'SVD'].copy()

# get only the ones with rmsProp on for svd
df_svd = df_svd[(df_svd['rmsProp'] == True)]

# Extract unique hyperparameters
batch_sizes = sorted(df['batch_size'].unique())
mlp_widths = sorted(df['mlp_width'].unique())
k_values = sorted(df_svd['k'].dropna().unique())
svd_lrs = sorted(df_svd['lr'].unique())
baseline_lrs = sorted(df_baseline['lr'].unique())
baseline_optimizers = df_baseline['optimizer'].unique().tolist()
rtols = sorted(df_svd['rtol'].dropna().unique())
optimizers_baseline = df_baseline['optimizer'].unique().tolist()
optimizers_baseline.remove('AdamW')
baseline_optimizers.remove("AdamW")

print("SVD Hyperparameters:")
print(f"  k_values: {k_values}")
print(f"  Learning rates: {svd_lrs}")
print(f"\nBaseline Hyperparameters:")
print(f"  Optimizers: {baseline_optimizers}")
print(f"  Learning rates: {baseline_lrs}")
print(f"\nMLP Widths: {mlp_widths}")
print(f"Batch Sizes: {batch_sizes}")
print(f"Rtol values: {rtols}")
print("Baseline optimizers:", optimizers_baseline)

SVD Hyperparameters:
  k_values: [4, 8, 16, 32, 48, 64, 96, 128]
  Learning rates: [1.0, 10.0, 50.0, 100.0]

Baseline Hyperparameters:
  Optimizers: ['Adam', 'RMSprop', 'SGD']
  Learning rates: [0.001]

MLP Widths: [32]
Batch Sizes: [64, 128, 256, 512]
Rtol values: [0.001, 0.01, 0.1]
Baseline optimizers: ['Adam', 'RMSprop', 'SGD']


## compare learning curves at different k, fix bs / rtol / lr

### train loss

In [7]:
for BS in batch_sizes:
    for RTOL in [1e-1,1e-2,1e-3]:
        for LR in svd_lrs:
            epochs = np.arange(1,21)
            epochs_val = np.arange(21)

            df_svd_sel = df_svd[(df_svd['batch_size'] == BS) & (df_svd['rtol'] == RTOL) & (df_svd['lr'] == LR)]
            avail_k = sorted(df_svd_sel['k'].dropna().unique().tolist())

            if len(df_svd_sel) == 0:
                continue
            fig, ax = plt.subplots(figsize=(8, 8))
            legend_entries = []

            cmaps = plt.get_cmap('Greys')
            grays = [cmaps(i) for i in np.linspace(0.3, 0.9, len(avail_k))]
            for kval in avail_k:
                df_svd_k = df_svd_sel[df_svd_sel['k'] == kval].iloc[0]
                if len(df_svd_k) == 0:
                    continue

                train_losses = get_loss_curve(df_svd_k, 'train')
                val_losses = get_loss_curve(df_svd_k, 'val')

                legend_entries.append(Line2D([],[],label=f"k={int(kval)}",color=grays[avail_k.index(kval)],lw=2))
                ax.plot(epochs, train_losses, color=grays[avail_k.index(kval)], lw=2, linestyle='-')
                #ax.plot(epochs_val, val_losses, color=grays[avail_k.index(kval)], lw=2,linestyle='--')
            
            legend_entries_standard = []
            for iopt, opt in enumerate(optimizers_baseline):
                df_base_sel = df_baseline[(df_baseline['batch_size'] == BS) & (df_baseline['optimizer'] == opt)]
                if len(df_base_sel) == 0:
                    continue
                df_base = df_base_sel.iloc[0]
                train_losses = get_loss_curve(df_base, 'train')
                val_losses = get_loss_curve(df_base, 'val')
                legend_entries_standard.append(Line2D([],[],label=f"{opt}",color=f"C{iopt}",lw=2))
                ax.plot(epochs, train_losses, f'C{iopt}-', lw=2)
                #ax.plot(epochs_val, val_losses, f'C{iopt}--', lw=2)

            ax.set_xlabel('Epoch')
            ax.set_ylabel('Training Loss')
            ax.set_title(f"bs = {BS}, rtol = {RTOL}, lr = {LR}")
            ax.set_yscale('log')
            ax.set_ylim(1e-2,3)

            leg1 = ax.legend(handles=legend_entries, loc='lower right',ncol=2,title='Sven')
            ax.add_artist(leg1)

            leg2 = ax.legend(handles=legend_entries_standard, loc='lower left',title='Baselines')

            plt.tight_layout()
            plt.savefig(PLOT_DIR / f'train_loss_kOverlay_bs{BS}_rtol{RTOL}_lr{LR}_withRmsProp.pdf')
            plt.close()

### val loss

In [8]:
for BS in batch_sizes:
    for RTOL in [1e-1,1e-2,1e-3]:
        for LR in svd_lrs:
            epochs = np.arange(1,21)
            epochs_val = np.arange(21)

            df_svd_sel = df_svd[(df_svd['batch_size'] == BS) & (df_svd['rtol'] == RTOL) & (df_svd['lr'] == LR)]
            avail_k = sorted(df_svd_sel['k'].dropna().unique().tolist())

            if len(df_svd_sel) == 0:
                continue
            fig, ax = plt.subplots(figsize=(8, 8))
            legend_entries = []

            cmaps = plt.get_cmap('Greys')
            grays = [cmaps(i) for i in np.linspace(0.3, 0.9, len(avail_k))]
            for kval in avail_k:
                df_svd_k = df_svd_sel[df_svd_sel['k'] == kval].iloc[0]
                if len(df_svd_k) == 0:
                    continue

                val_losses = get_loss_curve(df_svd_k, 'val')

                legend_entries.append(Line2D([],[],label=f"k={int(kval)}",color=grays[avail_k.index(kval)],lw=2))
                ax.plot(epochs_val, val_losses, color=grays[avail_k.index(kval)], lw=2, linestyle='-')
                #ax.plot(epochs_val, val_losses, color=grays[avail_k.index(kval)], lw=2,linestyle='--')
            
            legend_entries_standard = []
            for iopt, opt in enumerate(optimizers_baseline):
                df_base_sel = df_baseline[(df_baseline['batch_size'] == BS) & (df_baseline['optimizer'] == opt)]
                if len(df_base_sel) == 0:
                    continue
                df_base = df_base_sel.iloc[0]
                val_losses = get_loss_curve(df_base, 'val')
                legend_entries_standard.append(Line2D([],[],label=f"{opt}",color=f"C{iopt}",lw=2))
                ax.plot(epochs_val, val_losses, f'C{iopt}-', lw=2)
                #ax.plot(epochs_val, val_losses, f'C{iopt}--', lw=2)

            ax.set_xlabel('Epoch')
            ax.set_ylabel('Validation Loss')
            ax.set_title(f"bs = {BS}, rtol = {RTOL}, lr = {LR}")
            ax.set_yscale('log')
            ax.set_ylim(1e-2,3)

            leg1 = ax.legend(handles=legend_entries, loc='lower right',ncol=2,title='Sven')
            ax.add_artist(leg1)

            leg2 = ax.legend(handles=legend_entries_standard, loc='lower left',title='Baselines')

            plt.tight_layout()
            plt.savefig(PLOT_DIR / f'val_loss_kOverlay_bs{BS}_rtol{RTOL}_lr{LR}_withRmsProp.pdf')
            plt.close()

## compare learning curves at different rtol, fix bs / k / lr

### train loss

In [9]:
for BS in batch_sizes:
    for k in k_values:
        for LR in svd_lrs:
            epochs = np.arange(1,21)
            epochs_val = np.arange(21)

            df_svd_sel = df_svd[(df_svd['batch_size'] == BS) & (df_svd['k'] == k) & (df_svd['lr'] == LR)]
            avail_rtol = sorted(df_svd_sel['rtol'].dropna().unique().tolist())

            if len(df_svd_sel) == 0:
                continue
            fig, ax = plt.subplots(figsize=(8, 8))
            legend_entries = []

            cmaps = plt.get_cmap('Greys')
            grays = [cmaps(i) for i in np.linspace(0.3, 0.9, len(avail_rtol))]
            for rtol in avail_rtol:
                df_svd_k = df_svd_sel[df_svd_sel['rtol'] == rtol].iloc[0]
                if len(df_svd_k) == 0:
                    continue

                train_losses = get_loss_curve(df_svd_k, 'train')
                val_losses = get_loss_curve(df_svd_k, 'val')

                legend_entries.append(Line2D([],[],label=f"rtol={rtol}",color=grays[avail_rtol.index(rtol)],lw=2))
                ax.plot(epochs, train_losses, color=grays[avail_rtol.index(rtol)], lw=2, linestyle='-')
                #ax.plot(epochs_val, val_losses, color=grays[avail_rtol.index(rtol)], lw=2,linestyle='--')
            
            legend_entries_standard = []
            for iopt, opt in enumerate(optimizers_baseline):
                df_base_sel = df_baseline[(df_baseline['batch_size'] == BS) & (df_baseline['optimizer'] == opt)]
                if len(df_base_sel) == 0:
                    continue
                df_base = df_base_sel.iloc[0]
                train_losses = get_loss_curve(df_base, 'train')
                val_losses = get_loss_curve(df_base, 'val')
                legend_entries_standard.append(Line2D([],[],label=f"{opt}",color=f"C{iopt}",lw=2))
                ax.plot(epochs, train_losses, f'C{iopt}-', lw=2)
                #ax.plot(epochs_val, val_losses, f'C{iopt}--', lw=2)

            ax.set_xlabel('Epoch')
            ax.set_ylabel('Training Loss')
            ax.set_title(f"bs = {BS}, k = {k}, lr = {LR}")
            ax.set_yscale('log')
            ax.set_ylim(1e-2,3)

            leg1 = ax.legend(handles=legend_entries, loc='lower right',ncol=2,title='Sven')
            ax.add_artist(leg1)

            leg2 = ax.legend(handles=legend_entries_standard, loc='lower left',title='Baselines')

            plt.tight_layout()
            plt.savefig(PLOT_DIR / f'train_loss_rtolOverlay_bs{BS}_k{k}_lr{LR}_withRmsProp.pdf')
            plt.close()

### val loss

In [10]:
for BS in batch_sizes:
    for k in k_values:
        for LR in svd_lrs:
            epochs = np.arange(1,21)
            epochs_val = np.arange(21)

            df_svd_sel = df_svd[(df_svd['batch_size'] == BS) & (df_svd['k'] == k) & (df_svd['lr'] == LR)]
            avail_rtol = sorted(df_svd_sel['rtol'].dropna().unique().tolist())

            if len(df_svd_sel) == 0:
                continue
            fig, ax = plt.subplots(figsize=(8, 8))
            legend_entries = []

            cmaps = plt.get_cmap('Greys')
            grays = [cmaps(i) for i in np.linspace(0.3, 0.9, len(avail_rtol))]
            for rtol in avail_rtol:
                df_svd_k = df_svd_sel[df_svd_sel['rtol'] == rtol].iloc[0]
                if len(df_svd_k) == 0:
                    continue

                val_losses = get_loss_curve(df_svd_k, 'val')

                legend_entries.append(Line2D([],[],label=f"rtol={rtol}",color=grays[avail_rtol.index(rtol)],lw=2))
                ax.plot(epochs_val, val_losses, color=grays[avail_rtol.index(rtol)], lw=2, linestyle='-')
                #ax.plot(epochs_val, val_losses, color=grays[avail_rtol.index(rtol)], lw=2,linestyle='--')
            
            legend_entries_standard = []
            for iopt, opt in enumerate(optimizers_baseline):
                df_base_sel = df_baseline[(df_baseline['batch_size'] == BS) & (df_baseline['optimizer'] == opt)]
                if len(df_base_sel) == 0:
                    continue
                df_base = df_base_sel.iloc[0]
                val_losses = get_loss_curve(df_base, 'val')
                legend_entries_standard.append(Line2D([],[],label=f"{opt}",color=f"C{iopt}",lw=2))
                ax.plot(epochs_val, val_losses, f'C{iopt}-', lw=2)
                #ax.plot(epochs_val, val_losses, f'C{iopt}--', lw=2)

            ax.set_xlabel('Epoch')
            ax.set_ylabel('Validation Loss')
            ax.set_title(f"bs = {BS}, k = {k}, lr = {LR}")
            ax.set_yscale('log')
            ax.set_ylim(1e-2,3)

            leg1 = ax.legend(handles=legend_entries, loc='lower right',ncol=2,title='Sven')
            ax.add_artist(leg1)

            leg2 = ax.legend(handles=legend_entries_standard, loc='lower left',title='Baselines')

            plt.tight_layout()
            plt.savefig(PLOT_DIR / f'val_loss_rtolOverlay_bs{BS}_k{k}_lr{LR}_withRmsProp.pdf')
            plt.close()

### 2.2 Training Curves: Best Configurations

In [31]:
# Plot validation loss curves for best configuration of each optimizer
fig, ax = plt.subplots(figsize=(8, 8))

# Get best SVD run data
best_svd_idx = df_svd['final_val_loss'].idxmin()
best_svd_row = df.loc[best_svd_idx]

epochs = np.arange(20)

ax.plot(epochs, get_loss_curve(best_svd_row, 'train'), 'C0-',
        label=f"Sven (lr={best_svd_row['lr']}, k={int(best_svd_row['k'])}, bs={int(best_svd_row['batch_size'])})")

for i, opt in enumerate(baseline_optimizers):
    opt_df = df_baseline[df_baseline['optimizer'] == opt]
    best_idx = opt_df['final_val_loss'].idxmin()
    best_row = df.loc[best_idx]
    ax.plot(epochs, get_loss_curve(best_row, 'train'), f'C{i+1}-',
            label=f"{opt} (lr={best_row['lr']:.0e}, bs={int(best_row['batch_size'])})")

ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
#ax.set_title('Validation Loss (Best Configurations)')
ax.set_yscale('log')
ax.legend(loc='upper right',ncol=2)
ax.set_title("MNIST")
plt.ylim([4e-2,4])

plt.tight_layout()
plt.savefig(PLOT_DIR / 'train_loss_best.pdf')
#plt.savefig(PLOT_DIR / 'validation_loss_best.png')
plt.close()

In [32]:
# Plot validation loss curves for best configuration of each optimizer
fig, ax = plt.subplots(figsize=(8, 8))

# Get best SVD run data
best_svd_idx = df_svd['final_val_loss'].idxmin()
best_svd_row = df.loc[best_svd_idx]

epochs = np.arange(21)

ax.plot(epochs, get_loss_curve(best_svd_row, 'val'), 'C0-',
        label=f"Sven (lr={best_svd_row['lr']}, k={int(best_svd_row['k'])}, bs={int(best_svd_row['batch_size'])})")

for i, opt in enumerate(baseline_optimizers):
    opt_df = df_baseline[df_baseline['optimizer'] == opt]
    best_idx = opt_df['final_val_loss'].idxmin()
    best_row = df.loc[best_idx]
    ax.plot(epochs, get_loss_curve(best_row, 'val'), f'C{i+1}-',
            label=f"{opt} (lr={best_row['lr']:.0e}, bs={int(best_row['batch_size'])})")

ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Loss')
#ax.set_title('Validation Loss (Best Configurations)')
ax.set_yscale('log')
ax.legend(loc='upper right',ncol=2)
ax.set_title("MNIST")
plt.ylim([4e-2,4])

plt.tight_layout()
plt.savefig(PLOT_DIR / 'validation_loss_best.pdf')
#plt.savefig(PLOT_DIR / 'validation_loss_best.png')
plt.close()

In [51]:
from matplotlib.lines import Line2D
# Plot validation loss curves for best configuration of each optimizer
fig, ax = plt.subplots(figsize=(8, 8))
legend_labels = []

# Get best SVD run data
best_svd_idx = df_svd['final_val_loss'].idxmin()
best_svd_row = df.loc[best_svd_idx]
legend_labels.append(f"Sven (lr={best_svd_row['lr']}, k={int(best_svd_row['k'])}, bs={int(best_svd_row['batch_size'])})")

epochs = np.arange(1,21)
epochs_val = np.arange(21)

ax.plot(epochs, get_loss_curve(best_svd_row, 'train'), 'C0-')
ax.plot(epochs_val, get_loss_curve(best_svd_row, 'val'), 'C0--')

for i, opt in enumerate(baseline_optimizers):
    opt_df = df_baseline[df_baseline['optimizer'] == opt]
    best_idx = opt_df['final_val_loss'].idxmin()
    best_row = df.loc[best_idx]
    ax.plot(epochs, get_loss_curve(best_row, 'train'), f'C{i+1}-')
    ax.plot(epochs_val, get_loss_curve(best_row, 'val'), f'C{i+1}--')
    legend_labels.append(f"{opt} (lr={best_row['lr']:.0e}, bs={int(best_row['batch_size'])})")

ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
#ax.set_title('Validation Loss (Best Configurations)')
ax.set_yscale('log')
ax.set_title("MNIST")
plt.ylim([4e-2,4])

leg1 = ax.legend(handles=[Line2D([], [], color=f'C{i}', linestyle='-',label=legend_labels[i]) for i in range(len(legend_labels))],loc='upper right',ncol=2)
ax.add_artist(leg1)

leg2 = ax.legend(handles=[Line2D([], [], color='k', linestyle='-', label='Train'),
                   Line2D([], [], color='k', linestyle=':', label='Val')],
                  loc='lower left')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'train_val_loss_best.pdf')
#plt.savefig(PLOT_DIR / 'train_val_loss_best.png')
plt.close()

### 2.2.1 Training curves for best configurations with batchwise loss

In [34]:
# Plot training loss curves for best configuration of each optimizer
fig, ax = plt.subplots(figsize=(8, 8))

# Get best SVD run data
best_svd_idx = df_svd['final_val_loss'].idxmin()
best_svd_row = df_svd.loc[best_svd_idx]

epochs = np.linspace(0, 20, len(best_svd_row['losses']['train_batch']))

ax.plot(epochs, get_loss_curve(best_svd_row, 'train_batch'), 'C0-', 
        label=f"Sven (lr={best_svd_row['lr']}, k={int(best_svd_row['k'])}, bs={int(best_svd_row['batch_size'])})")

for i, opt in enumerate(baseline_optimizers):
    opt_df = df_baseline[df_baseline['optimizer'] == opt]
    best_idx = opt_df['final_val_loss'].idxmin()
    best_row = opt_df.loc[best_idx]
    epochs = np.linspace(0, 20, len(best_row['losses']['train_batch']))
    ax.plot(epochs, get_loss_curve(best_row, 'train_batch'), f'C{i+1}-',
            label=f"{opt} (lr={best_row['lr']:.0e}, bs={int(best_row['batch_size'])})")

ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
#ax.set_title('Training Loss (Best Configurations)')
ax.set_yscale('log')
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'training_loss_best_allBatches.pdf')
#plt.savefig(PLOT_DIR / 'training_loss_best_allBatches.png')
plt.close()

### 2.3 Training curves for best configurations at each batch size

In [53]:
# Compare training loss at a specific batch size (e.g., 64)
for comparison_bs in [64,128]:
        print(f"bs = {comparison_bs}")
        fig, ax = plt.subplots(figsize=(8, 8))

        # Filter data
        svd_bs = df_svd[df_svd['batch_size'] == comparison_bs]
        baseline_bs = df_baseline[df_baseline['batch_size'] == comparison_bs]

        # Find best SVD config at this batch size
        best_svd_at_bs = svd_bs.loc[svd_bs['final_val_loss'].idxmin()]

        epochs = np.arange(1,21)
        epochs_val = np.arange(21)

        legend_entries = []

        # Training curves
        legend_entries.append(Line2D([],[],label=f"Sven (lr={best_svd_at_bs['lr']}, k={int(best_svd_at_bs['k'])})",color="C0",linewidth=2))
        ax.plot(epochs, get_loss_curve(best_svd_at_bs, 'train'), 'C0-', lw=2)
        ax.plot(epochs_val,get_loss_curve(best_svd_at_bs,'val'),'C0--',lw=2)

        for i, opt in enumerate(baseline_optimizers):
                opt_data = baseline_bs[baseline_bs['optimizer'] == opt]
                best = opt_data.loc[opt_data['final_val_loss'].idxmin()]
                ax.plot(epochs, get_loss_curve(best, 'train'), f'C{i+1}-')
                ax.plot(epochs_val,get_loss_curve(best,'val'),f'C{i+1}--')
                legend_entries.append(Line2D([],[],label=f"{opt} (lr={best['lr']:.0e})",color=f'C{i+1}'))

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_title(f'MNIST, Batch Size = {comparison_bs}')
        ax.set_yscale('log')
        ax.set_ylim(3e-2,5)
        
        leg1 = ax.legend(handles=legend_entries, loc='upper right',ncol=2)
        ax.add_artist(leg1)
        
        leg2_entries = [Line2D([], [], color='k', linestyle='-', label='Train'),
                        Line2D([], [], color='k', linestyle='--', label='Val')]
        leg2 = ax.legend(handles=leg2_entries, loc='lower left')

        plt.tight_layout()
        plt.savefig(PLOT_DIR / f'best_train_val_loss_bs{comparison_bs}.pdf')
        #plt.savefig(PLOT_DIR / f'best_train_val_loss_bs{comparison_bs}.png')
        plt.close()

bs = 64
bs = 128


### Compare learning at different batch sizes, SVD

In [38]:
fig, ax = plt.subplots(figsize=(8, 8))
legend_entries = []

for i,comparison_bs in enumerate([64,128]):
    # Filter data
    svd_bs = df_svd[df_svd['batch_size'] == comparison_bs]

    # Find best SVD config at this batch size
    best_svd_at_bs = svd_bs.loc[svd_bs['final_val_loss'].idxmin()]

    epochs = np.arange(1,21)
    epochs_val = np.arange(21)

    # Training curves
    legend_entries.append(Line2D([],[],label=f"bs = {comparison_bs} (lr={best_svd_at_bs['lr']}, k={int(best_svd_at_bs['k'])})",color=f"C{i}",linewidth=2))
    ax.plot(epochs, get_loss_curve(best_svd_at_bs, 'train'), f'C{i}-', lw=2)
    ax.plot(epochs_val,get_loss_curve(best_svd_at_bs,'val'),f'C{i}--',lw=2)


ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_yscale('log')
ax.set_ylim(1e-4,2)

leg1 = ax.legend(handles=legend_entries, loc='upper right',ncol=2)
ax.add_artist(leg1)

leg2_entries = [Line2D([], [], color='k', linestyle='-', label='Train'),
                Line2D([], [], color='k', linestyle='--', label='Val')]
leg2 = ax.legend(handles=leg2_entries, loc='lower left')

plt.tight_layout()
plt.savefig(PLOT_DIR / f'best_train_val_loss_bsScan.pdf')
#plt.savefig(PLOT_DIR / f'best_train_val_loss_bsScan.png')
plt.close()

In [41]:
batch_sizes = [64,128]

In [45]:
lrs = sorted(df_svd['lr'].unique().tolist())
k_fracs = sorted(df_svd['k_fraction'].unique().tolist())
for KFRAC in k_fracs:
    for LR in lrs:
        try:
            df_svd_sel = df_svd[(df_svd['k_fraction'] == KFRAC) & (df_svd['lr'] == LR)]
            fig, ax = plt.subplots(figsize=(8, 8))
            legend_entries = []

            for i,comparison_bs in enumerate(batch_sizes):
                # Filter data
                svd_bs = df_svd_sel[df_svd_sel['batch_size'] == comparison_bs]

                # Find best SVD config at this batch size
                best_svd_at_bs = svd_bs.loc[svd_bs['final_val_loss'].idxmin()]

                epochs = np.arange(1,21)
                epochs_val = np.arange(21)

                # Training curves
                legend_entries.append(Line2D([],[],label=f"bs = {comparison_bs}",color=f"C{i}",linewidth=2))
                ax.plot(epochs, get_loss_curve(best_svd_at_bs, 'train'), f'C{i}-', lw=2)
                ax.plot(epochs_val,get_loss_curve(best_svd_at_bs,'val'),f'C{i}--',lw=2)


            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.set_title(f'k/bs = {KFRAC}, lr = {LR}')
            ax.set_yscale('log')
            ax.set_ylim(1e-4,2)

            leg1 = ax.legend(handles=legend_entries, loc='upper right')
            ax.add_artist(leg1)

            leg2_entries = [Line2D([], [], color='k', linestyle='-', label='Train'),
                            Line2D([], [], color='k', linestyle='--', label='Val')]
            leg2 = ax.legend(handles=leg2_entries, loc='lower left')

            plt.tight_layout()
            plt.savefig(PLOT_DIR / f'train_val_loss_bsScan_kf{KFRAC}_lr{LR}.pdf')
            #plt.savefig(PLOT_DIR / f'train_val_loss_bsScan_kf{KFRAC}_lr{LR}.png')
            plt.close()
        except:
            plt.close()
                

### compare batchwise train loss at different batch sizes, SVD

In [46]:
legend_entries = []
lrs = sorted(df_svd['lr'].unique().tolist())

for LR in lrs:
    fig, ax = plt.subplots(figsize=(8, 8))
    for i,comparison_bs in enumerate(batch_sizes):
        # Filter data
        svd_bs = df_svd[(df_svd['batch_size'] == comparison_bs) & (df_svd['lr'] == LR)]

        # Find best SVD config at this batch size
        best_svd_at_bs = svd_bs.loc[svd_bs['final_val_loss'].idxmin()]

        # Training curves
        ax.plot(get_loss_curve(best_svd_at_bs, 'train_batch'), f'C{i}-', lw=2, label=f"bs = {comparison_bs} (k={int(best_svd_at_bs['k'])})")

    ax.set_xlabel('Batch')
    ax.set_ylabel('Training Loss')
    ax.set_title(f'lr = {LR}')
    ax.set_yscale('log')
    ax.set_ylim(1e-4,10)
    ax.set_xlim(0,200)

    plt.legend(loc='upper right')

    plt.tight_layout()
    plt.savefig(PLOT_DIR / f'svd_best_train_loss_byBatch_lr{LR}.pdf')
    #plt.savefig(PLOT_DIR / f'svd_best_train_loss_byBatch_lr{LR}.png')
    plt.close()

In [48]:
lrs = sorted(df_svd['lr'].unique().tolist())
k_fracs = sorted(df_svd['k_fraction'].unique().tolist())
for KFRAC in k_fracs:
    for LR in lrs:
        try:
            df_svd_sel = df_svd[(df_svd['k_fraction'] == KFRAC) & (df_svd['lr'] == LR)]
            fig, ax = plt.subplots(figsize=(8, 8))
            legend_entries = []

            for i,comparison_bs in enumerate(batch_sizes):
                # Filter data
                svd_bs = df_svd_sel[df_svd_sel['batch_size'] == comparison_bs]

                # Find best SVD config at this batch size
                best_svd_at_bs = svd_bs.loc[svd_bs['final_val_loss'].idxmin()]

                # Training curves
                ax.plot(get_loss_curve(best_svd_at_bs, 'train_batch'), f'C{i}-', lw=2, label=f"bs = {comparison_bs}")


            ax.set_xlabel('Batch')
            ax.set_ylabel('Training Loss')
            ax.set_title(f'k/bs = {KFRAC}, lr = {LR}')
            ax.set_yscale('log')
            ax.set_ylim(1e-4,10)
            ax.set_xlim(0,200)

            ax.legend(loc='upper right')

            plt.tight_layout()
            plt.savefig(PLOT_DIR / f'svd_train_loss_byBatch_bsScan_kf{KFRAC}_lr{LR}.pdf')
            #plt.savefig(PLOT_DIR / f'svd_train_loss_byBatch_bsScan_kf{KFRAC}_lr{LR}.png')
            plt.close()
        except:
            plt.close()
                

### Fix batch size and LR, compare curves at different k fractions

In [49]:
batch_sizes = sorted(df_svd['batch_size'].unique().tolist())
lrs = sorted(df_svd['lr'].unique().tolist())
k_fracs = sorted(df_svd['k_fraction'].unique().tolist())
for BS in batch_sizes:
    for LR in lrs:
        try:
            df_svd_sel = df_svd[(df_svd['batch_size'] == BS) & (df_svd['lr'] == LR)]
            fig, ax = plt.subplots(figsize=(8, 8))
            legend_entries = []

            for i,comparison_kfrac in enumerate(k_fracs):
                # Filter data
                svd_kf = df_svd_sel[df_svd_sel['k_fraction'] == comparison_kfrac]
                assert len(svd_kf) == 1
                svd_kf = svd_kf.iloc[0]
                ax.plot(svd_kf['losses']['train_batch'], f'C{i}-', lw=2, label=f"k/bs = {comparison_kfrac} ({int(comparison_kfrac*BS)})")
            
            ax.set_xlabel('Batch')
            ax.set_ylabel('Training Loss')
            ax.set_title(f'bs = {BS}, lr = {LR}')
            ax.set_yscale('log')
            ax.set_ylim(1e-4,10)
            ax.set_xlim(0,250)

            ax.legend(loc='upper right')

            plt.tight_layout()
            plt.savefig(PLOT_DIR / f'svd_train_loss_byBatch_kfScan_bs{BS}_lr{LR}.pdf')
            #plt.savefig(PLOT_DIR / f'svd_train_loss_byBatch_kfScan_bs{BS}_lr{LR}.png')
            plt.close()
        except:
            plt.close()

### Fix batch size and k fraction, compare curves at different LR

In [50]:
batch_sizes = sorted(df_svd['batch_size'].unique().tolist())
lrs = sorted(df_svd['lr'].unique().tolist())
k_fracs = sorted(df_svd['k_fraction'].unique().tolist())
for BS in batch_sizes:
    for KF in k_fracs:
        try:
            df_svd_sel = df_svd[(df_svd['batch_size'] == BS) & (df_svd['k_fraction'] == KF)]
            fig, ax = plt.subplots(figsize=(8, 8))
            legend_entries = []

            for i,comparison_lr in enumerate(lrs):
                # Filter data
                svd_kf = df_svd_sel[df_svd_sel['lr'] == comparison_lr]
                assert len(svd_kf) == 1
                svd_kf = svd_kf.iloc[0]
                ax.plot(svd_kf['losses']['train_batch'], f'C{i}-', lw=2, label=f"lr = {comparison_lr}")
            
            ax.set_xlabel('Batch')
            ax.set_ylabel('Training Loss')
            ax.set_title(f'bs = {BS}, k/bs = {KF}')
            ax.set_yscale('log')
            ax.set_ylim(1e-4,10)
            ax.set_xlim(0,250)

            ax.legend(loc='upper right')

            plt.tight_layout()
            plt.savefig(PLOT_DIR / f'svd_train_loss_byBatch_lrScan_bs{BS}_kf{KF}.pdf')
            #plt.savefig(PLOT_DIR / f'svd_train_loss_byBatch_lrScan_bs{BS}_kf{KF}.png')
            plt.close()
        except:
            plt.close()

## 3. Hyperparameter Sensitivity Analysis

Analyze how SVD optimizer performance depends on:
- Learning rate
- Batch size
- Number of singular values (k)

### 3.1 Heatmap: Final Loss vs Hyperparameters

In [ ]:
# Create heatmap of final validation loss for SVD optimizer
# For each batch size, show k_fraction vs learning rate

fig, axes = plt.subplots(1, len(batch_sizes), figsize=(3*len(batch_sizes), 3.5))
if len(batch_sizes) == 1:
    axes = [axes]

for ax, bs in zip(axes, batch_sizes):
    # Create pivot table
    bs_data = df_svd[df_svd['batch_size'] == bs]
    pivot = bs_data.pivot_table(
        values='final_val_loss', 
        index='k_fraction', 
        columns='lr',
        aggfunc='first'
    )
    
    # Plot heatmap (viridis: dark for low values, light for high values)
    im = ax.imshow(np.log10(pivot.values), aspect='auto', cmap='viridis',
                   vmin=-7, vmax=-3)
    
    # Labels
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'{lr}' for lr in pivot.columns], rotation=45)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{kf}' for kf in pivot.index])
    ax.set_xlabel('Learning Rate')
    ax.set_ylabel('k / bs')
    ax.set_title(f'Batch Size = {bs}')

    if ax != axes[0]:
        ax.set_ylabel('')
        ax.set_yticklabels([])

# Colorbar
fig.subplots_adjust(right=0.85)
cbar_ax = fig.add_axes([0.88, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('log10(Validation Loss)')

#plt.suptitle('SVD Optimizer: Final Validation Loss', fontsize=14, y=1.02)
plt.savefig(PLOT_DIR / 'svd_heatmap_by_batchsize.pdf', bbox_inches='tight')
#plt.savefig(PLOT_DIR / 'svd_heatmap_by_batchsize.png', bbox_inches='tight')
plt.close()

### 3.2 Learning Rate Sensitivity

In [ ]:
for bs in batch_sizes:
    # Learning rate sensitivity at different k_fractions
    fig,ax = plt.subplots(1, 1, figsize=(8, 8))
    bs_data = df_svd[df_svd['batch_size'] == bs]

    # Plot for each k_fraction
    for i, kf in enumerate(k_fractions):
        kf_data = bs_data[bs_data['k_fraction'] == kf].sort_values('lr')
        ax.plot(kf_data['lr'], kf_data['final_val_loss'], 'o-', 
                label=f'k/bs = {kf} (k={int(kf*bs)})', color=f'C{i}')

    ax.set_xlabel('Learning Rate')
    ax.set_ylabel('Final Validation Loss')
    ax.set_title(f'Learning Rate Sensitivity (Batch Size = {bs})')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.legend()

    plt.tight_layout()
    plt.savefig(PLOT_DIR / f'lr_sensitivity_bs{bs}.pdf')
    #plt.savefig(PLOT_DIR / f'lr_sensitivity_bs{bs}.png')
    plt.close()

### 3.3 Batch Size Effect

In [ ]:
for fixed_lr in svd_lrs:
    for fixed_kf in k_fractions:
        # Batch size effect on SVD optimizer
        fig, ax = plt.subplots(1, 1, figsize=(8, 8))

        # SVD: vary batch size
        data = df_svd[(df_svd['lr'] == fixed_lr) & (df_svd['k_fraction'] == fixed_kf)]
        data = data.sort_values('batch_size')
        ax.plot(data['batch_size'], data['final_val_loss'], 'o-', label=f'Sven')

        # Best baseline per batch size
        for i, opt in enumerate(baseline_optimizers):
            opt_data = df_baseline[df_baseline['optimizer'] == opt]
            opt_best_per_bs = opt_data.groupby('batch_size')['final_val_loss'].min()
            ax.plot(opt_best_per_bs.index, opt_best_per_bs.values, 's--', 
                    label=f'{opt} (best)', color=f'C{i+1}')

        ax.set_xlabel('Batch Size')
        ax.set_ylabel('Final Validation Loss')
        ax.set_title(f'k/bs = {fixed_kf}, SVD lr = {fixed_lr}')
        ax.set_xscale('log', base=2)
        ax.set_yscale('log')
        ax.set_xticks(batch_sizes)
        ax.set_xticklabels(batch_sizes)
        ax.legend()

        plt.tight_layout()
        plt.savefig(PLOT_DIR / f'batch_size_effect_lr{fixed_lr}_kf{fixed_kf}.pdf')
        #plt.savefig(PLOT_DIR / f'batch_size_effect_lr{fixed_lr}_kf{fixed_kf}.png')
        plt.close()

### 3.4 Number of Singular Values (k) Effect

In [ ]:
# Plot k vs final loss for different batch sizes
svd_lrs = sorted(df_svd['lr'].unique().tolist())
for fixed_lr in svd_lrs:
    fig, ax = plt.subplots(figsize=(8, 8))
    for bs in batch_sizes:
        data = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == fixed_lr)]
        data = data.sort_values('k')
        ax.plot(data['k'], data['final_val_loss'], 'o-', label=f'bs={bs}')

    ax.set_xlabel('k (Number of Singular Values)')
    ax.set_ylabel('Final Validation Loss')
    ax.set_title(f'lr = {fixed_lr}')
    ax.set_xscale('log', base=2)
    ax.set_yscale('log')
    ax.legend()

    plt.tight_layout()
    plt.savefig(PLOT_DIR / f'k_effect_lr_{fixed_lr}.pdf')
    #plt.savefig(PLOT_DIR / f'k_effect_lr_{fixed_lr}.png')
    plt.close()

In [ ]:
# Effect of k_fraction (normalized by batch size)
svd_lrs = sorted(df_svd['lr'].unique().tolist())
for fixed_lr in svd_lrs:
    fig, ax = plt.subplots(figsize=(8, 8))
    for bs in batch_sizes:
        data = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == fixed_lr)]
        data = data.sort_values('k_fraction')
        ax.plot(data['k_fraction'], data['final_val_loss'], 'o-', label=f'bs={bs}')

    ax.set_xlabel('k / bs')
    ax.set_ylabel('Final Validation Loss')
    ax.set_title(f'lr = {fixed_lr}')
    ax.set_yscale('log')
    ax.legend()

    plt.tight_layout()
    plt.savefig(PLOT_DIR / f'k_fraction_effect_lr_{fixed_lr}.pdf')
    #plt.savefig(PLOT_DIR / f'k_fraction_effect_lr_{fixed_lr}.png')
    plt.close()

## 4. Singular Value Analysis

Analyze the singular value spectrum of the Jacobian during training.

### Num nonzero SVs, fix bs and lr, overlay k_frac

In [ ]:
batch_sizes = sorted(df_svd['batch_size'].unique().tolist())
svd_lrs = sorted(df_svd['lr'].unique().tolist())
k_fractions = sorted(df_svd['k_fraction'].unique().tolist())

smooth = 20
num_batches_plot = 200

for BS in batch_sizes:
    for LR in svd_lrs:
        fig, ax = plt.subplots(figsize=(8, 8))
        
        df_svd_sel = df_svd[(df_svd['batch_size'] == BS) & (df_svd['lr'] == LR)]
        legend_entries = []
        for i,comparison_kfrac in enumerate(k_fractions):
            # Filter data
            svd_kf = df_svd_sel[df_svd_sel['k_fraction'] == comparison_kfrac]
            assert len(svd_kf) == 1
            row = svd_kf.iloc[0]
            num_nonzero = row['svd_info']['num_nonzero_svs']
            num_epoch = len(row['losses']['train'])
            x = np.linspace(0, num_epoch, len(num_nonzero))
            ax.plot(x[:num_batches_plot], num_nonzero[:num_batches_plot], f'C{i}-', lw=2,alpha=0.25)
            n_nonzero_smooth = sliding_average(num_nonzero, window=smooth)
            x_smooth = x[smooth-1:]
            ax.plot(x_smooth[:num_batches_plot], n_nonzero_smooth[:num_batches_plot], f'C{i}--', lw=2)
            legend_entries.append(Line2D([],[],label=f"k/bs = {comparison_kfrac} ({int(comparison_kfrac*BS)})",color=f"C{i}"))
        
        ax.set_xlabel('Epoch')
        ax.set_ylabel('# Nonzero Singular Values')
        ax.set_title(f'bs = {BS}, lr = {LR}')
        leg1 = ax.legend(handles=legend_entries)
        ax.add_artist(leg1)
        leg2 = ax.legend(handles=[Line2D([], [], color='k', linestyle='-', label='Batchwise'),
                           Line2D([], [], color='k', linestyle='--', label=f'Smoothed (window={smooth})')],
                          loc='lower left')

        plt.tight_layout()
        plt.savefig(PLOT_DIR / f'num_nonzero_svs_bs{BS}_lr{LR}_{num_batches_plot}batches_smooth{smooth}.pdf')
        #plt.savefig(PLOT_DIR / f'num_nonzero_svs_bs{BS}_lr{LR}_{num_batches_plot}batches_smooth{smooth}.png')
        plt.close()

In [ ]:
batch_sizes = sorted(df_svd['batch_size'].unique().tolist())
svd_lrs = sorted(df_svd['lr'].unique().tolist())
k_fractions = sorted(df_svd['k_fraction'].unique().tolist())

smooth = 20

for BS in batch_sizes:
    for LR in svd_lrs:
        fig, ax = plt.subplots(figsize=(8, 8))
        
        df_svd_sel = df_svd[(df_svd['batch_size'] == BS) & (df_svd['lr'] == LR)]
        legend_entries = []
        for i,comparison_kfrac in enumerate(k_fractions):
            # Filter data
            svd_kf = df_svd_sel[df_svd_sel['k_fraction'] == comparison_kfrac]
            assert len(svd_kf) == 1
            row = svd_kf.iloc[0]
            num_nonzero = row['svd_info']['num_nonzero_svs']
            num_epoch = len(row['losses']['train'])
            x = np.linspace(0, num_epoch, len(num_nonzero))
            ax.plot(x, num_nonzero, f'C{i}-', lw=2,alpha=0.25)
            n_nonzero_smooth = sliding_average(num_nonzero, window=smooth)
            x_smooth = x[smooth-1:]
            ax.plot(x_smooth, n_nonzero_smooth, f'C{i}--', lw=2)
            legend_entries.append(Line2D([],[],label=f"k/bs = {comparison_kfrac} ({int(comparison_kfrac*BS)})",color=f"C{i}"))
        
        ax.set_xlabel('Epoch')
        ax.set_ylabel('# Nonzero Singular Values')
        ax.set_title(f'bs = {BS}, lr = {LR}')
        leg1 = ax.legend(handles=legend_entries)
        ax.add_artist(leg1)
        leg2 = ax.legend(handles=[Line2D([], [], color='k', linestyle='-', label='Batchwise'),
                           Line2D([], [], color='k', linestyle='--', label=f'Smoothed (window={smooth})')],
                          loc='lower left')

        plt.tight_layout()
        plt.savefig(PLOT_DIR / f'num_nonzero_svs_bs{BS}_lr{LR}_smooth{smooth}.pdf')
        #plt.savefig(PLOT_DIR / f'num_nonzero_svs_bs{BS}_lr{LR}_smooth{smooth}.png')
        plt.close()

In [ ]:
batch_sizes = sorted(df_svd['batch_size'].unique().tolist())
svd_lrs = sorted(df_svd['lr'].unique().tolist())
k_fractions = sorted(df_svd['k_fraction'].unique().tolist())

for BS in batch_sizes:
    for LR in svd_lrs:
        fig, ax = plt.subplots(figsize=(8, 8))
        ax2 = ax.twinx()  # Second y-axis for training loss
        
        df_svd_sel = df_svd[(df_svd['batch_size'] == BS) & (df_svd['lr'] == LR)]
        legend_entries = []
        for i,comparison_kfrac in enumerate(k_fractions):
            # Filter data
            svd_kf = df_svd_sel[df_svd_sel['k_fraction'] == comparison_kfrac]
            assert len(svd_kf) == 1
            row = svd_kf.iloc[0]
            num_nonzero = row['svd_info']['num_nonzero_svs']
            num_epoch = len(row['losses']['train'])
            x = np.linspace(0, num_epoch, len(num_nonzero))
            smooth_fraction = 0.1 # one-tenth of an epoch
            smooth = max(1, int(smooth_fraction * (len(num_nonzero) / num_epoch)))
            n_nonzero_smooth = sliding_average(num_nonzero, window=smooth)
            x_smooth = x[smooth-1:]
            ax.plot(x_smooth, n_nonzero_smooth, f'C{i}-', lw=2)
            
            # Plot training loss on second y-axis
            train_loss = row['losses']['train']
            epochs_train = np.linspace(1, len(train_loss), len(train_loss))
            train_loss_batch = row['losses']['train_batch'] 
            train_loss_smooth = sliding_average(train_loss_batch, window=smooth)
            ax2.plot(x_smooth, train_loss_smooth, f'C{i}--', lw=2, alpha=0.7)
            
            legend_entries.append(Line2D([],[],label=f"k/bs = {comparison_kfrac} ({int(comparison_kfrac*BS)})",color=f"C{i}"))

        
        ax.set_xlabel('Epoch')
        ax.set_ylabel('# Nonzero Singular Values')
        ax2.set_ylabel('Training Loss')
        ax2.set_yscale('log')
        ax.set_title(f'bs = {BS}, lr = {LR}')
        leg1 = ax.legend(handles=legend_entries, loc='upper right',ncol=2)
        ax.add_artist(leg1)
        leg2 = ax.legend(handles=[Line2D([], [], color='k', linestyle='-', label='# Nonzero SVs'),
                           Line2D([], [], color='k', linestyle='--', label='Training Loss')],
                          loc='lower left')

        plt.tight_layout()
        plt.savefig(PLOT_DIR / f'num_nonzero_svs_trainloss_bs{BS}_lr{LR}_smoothFrac{smooth_fraction}.pdf')
        #plt.savefig(PLOT_DIR / f'num_nonzero_svs_trainloss_bs{BS}_lr{LR}_smoothFrac{smooth_fraction}.png')
        plt.close()

### Maximum and minimum SVs, fix batch and lr

In [ ]:
batch_sizes = sorted(df_svd['batch_size'].unique().tolist())
svd_lrs = sorted(df_svd['lr'].unique().tolist())
k_fractions = sorted(df_svd['k_fraction'].unique().tolist())

smooth = 20

for BS in batch_sizes:
    for LR in svd_lrs:
        fig, ax = plt.subplots(figsize=(8, 8))
        
        df_svd_sel = df_svd[(df_svd['batch_size'] == BS) & (df_svd['lr'] == LR)]
        legend_entries = []
        for i,comparison_kfrac in enumerate(k_fractions):
            # Filter data
            svd_kf = df_svd_sel[df_svd_sel['k_fraction'] == comparison_kfrac]
            assert len(svd_kf) == 1
            row = svd_kf.iloc[0]
            svs = row['svd_info']['svs']
            max_sv = [np.max(s) for s in svs]
            num_epoch = len(row['losses']['train'])
            x = np.linspace(0, num_epoch, len(svs))
            ax.plot(x, max_sv, f'C{i}-', lw=2,alpha=0.25)
            max_sv_smooth = sliding_average(max_sv, window=smooth)
            x_smooth = x[smooth-1:]
            ax.plot(x_smooth, max_sv_smooth, f'C{i}--', lw=2)
            legend_entries.append(Line2D([],[],label=f"k/bs = {comparison_kfrac} ({int(comparison_kfrac*BS)})",color=f"C{i}"))
        
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Maximum Singular Value')
        ax.set_title(f'bs = {BS}, lr = {LR}')
        ax.set_yscale('log')
        leg1 = ax.legend(handles=legend_entries)
        ax.add_artist(leg1)
        leg2 = ax.legend(handles=[Line2D([], [], color='k', linestyle='-', label='Batchwise'),
                           Line2D([], [], color='k', linestyle='--', label=f'Smoothed (window={smooth})')],
                          loc='lower left')

        plt.tight_layout()
        plt.savefig(PLOT_DIR / f'max_sv_bs{BS}_lr{LR}_smooth{smooth}.pdf')
        #plt.savefig(PLOT_DIR / f'max_sv_bs{BS}_lr{LR}_smooth{smooth}.png')
        plt.close()